# MIRAGE FlowPic Classification for Network Traffic Analysis
di Mario Gabriele Carofano

Questo notebook implementa una pipeline completa di **Traffic Classification** utilizzando la rappresentazione **FlowPic** sul dataset MIRAGE, dalla generazione del dataset fino alla valutazione finale dei modelli.

### Panoramica

Il workflow seguito è:

1. **Setup ambiente e riproducibilità**
	- Import librerie, costanti e funzioni custom.
	- Impostazione seed (`random`, `numpy`, `torch`) e scelta automatica del device (`mps` / `cuda` / `cpu`).

2. **Generazione del dataset (opzionale)**

	- Se `USE_PRECOMPUTED_DATASET` è impostato su `False`: il notebook genera il dataset FlowPic da zero utilizzando le funzioni custom implementate nel modulo `traffic_converter.py`.

	- Se `USE_PRECOMPUTED_DATASET` è impostato su `True`: il notebook carica da disco un dataset FlowPic già pre-calcolato.

3. **Preparazione dei dati**
	...

4. **Model selection**
	...

5. **Fase di training**
	...

6. **Fase di valutazione (su test set)**
	- Calcolo di **accuracy**, **confusion matrix** e **classification report**.

### Dataset (configurabile)

- **Formato**: `Tuple[np.ndarray, pd.DataFrame]`, dove:
	- `np.ndarray`: Dataset di istogrammi 2D delle sessioni di traffico, di shape `(N, 1, D, D)`, dove `N` è il numero di finestre temporali valide e `D` è la dimensione dell'istogramma, calcolata in base ai parametri `MTU` e `BIN_SIZE`.
	- `pd.DataFrame`: Metadati dei flussi validi, con le colonne `"FlowID"`, `"DatasetID"` e `"Label"`.
- **Shape dell'input**: `(N, 1, D, D)` — istogrammi 2D FlowPic delle sessioni di traffico.
- **Dimensione dell'istogramma (`D`)**: Calcolata in base alle costanti `MTU` e `BIN_SIZE`.

### Output
...

---

## Setup ambiente e riproducibilità

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing constant values
import importlib
import sys
sys.path.insert(1, '../src/')
import constants
importlib.reload(constants)

# Data loading and saving
from pathlib import Path
import pickle
import os

# Importing traffic converter functions
from traffic_converter import *

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data preprocessing and evaluation
from preprocessing_functions import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Machine Learning and Quantum ML
from training_functions import *
from model_selection import *
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import pennylane as qml

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Other utilities
import copy
import datetime
import random
import time

In [ ]:
#	MACROS
#   ####################################################################    #

importlib.reload(constants)
from constants import RANDOM_SEED

#   ####################################################################    #

# 1. Configurazione del seed per la riproducibilità.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

# 2. Configurazione di PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. Configurazione di PyTorch per la riproducibilità su GPU (CUDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 4. Configurazione del dispositivo (CPU o GPU)
if torch.backends.mps.is_available():
	DEVICE = torch.device("mps")
elif torch.cuda.is_available():
	DEVICE = torch.device("cuda")
else:
	DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE.type}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#	CLASSES
#	####################################################################    #

class FlowPicDataset(Dataset):
	""" Custom PyTorch Dataset per il caricamento e la preparazione di istogrammi
    2D FlowPic (traffico di rete) e dei relativi metadati.

	A differenza della versione per sequenze di pacchetti (pensata per
    CNN1D), qui i dati sono già nel formato immagine (N, C, H, W)
    adatto a CNN2D senza alcuna trasposizione (o permutazione).

	Caratteristiche principali:
    -	Encoding automatico delle etichette testuali (Label) tramite
	sklearn.LabelEncoder, riusabile tra split train/val/test.
    -	Trasferimento "lazy" dei tensori sul device per singolo batch,
	per evitare Out-Of-Memory su GPU/MPS con dataset di grandi
	dimensioni (es. 93300 x 1 x 150 x 150 ~ 8-17 GB in RAM).
    -	Accesso ai metadati originali (FlowID, DatasetID, Label) per
	ogni campione, utile per debug/analisi degli errori.
    -	Supporto opzionale per trasformazioni (normalizzazione, augmentation).
	"""
		
	def __init__(
			self, X_raw, y_raw,
			device, dtype=None, transform=None,
			lazy_device_transfer=True
		):
		"""Inizializza il dataset FlowPic.

		Args:
			X_raw (np.ndarray): Istogrammi 2D, shape (N, 1, H, W).
			y_raw (np.ndarray): Etichette già codificate numericamente, shape (N,).
			device (torch.device): Device usato per il trasferimento lazy dei batch.
			dtype (torch.dtype, optional): Tipo dei tensori immagine.
			Se None, float32 su MPS (che non supporta float64),
			float64 altrove. Defaults to None.
			transform (Callable, optional): Funzione applicata a ciascun tensore
			immagine in __getitem__ (es. normalizzazione). Defaults to None.
			lazy_device_transfer (bool, optional): Se True, i dati restano in CPU
			(pinnati se CUDA) e vengono spostati sul device solo per il batch
			richiesto, evitando OOM su GPU/MPS. Defaults to True.
		"""

		super().__init__()

		if len(X_raw) != len(y_raw):
			raise ValueError(
				f"X_raw ({len(X_raw)}) e y_raw ({len(y_raw)}) devono avere "
				f"lo stesso numero di campioni."
			)

		# MPS non supporta i tensori float64.
		self.device = device
		self.dtype = dtype or (
			torch.float32 if device.type == "mps" else torch.float64
		)

		self.lazy_device_transfer = lazy_device_transfer
		self.transform = transform

		X_t = torch.from_numpy(np.asarray(X_raw)).to(self.dtype).contiguous()
		y_t = torch.from_numpy(np.asarray(y_raw)).long()

		if self.lazy_device_transfer:
			# Tiene i dati in CPU; il trasferimento avviene per batch in __getitem__.
			self.X = X_t.pin_memory() if self.device.type == "cuda" else X_t
			self.y = y_t
		else:
			# Trasferisce tutto sul device (attenzione a OOM su GPU/MPS).
			self.X = X_t.to(self.device, non_blocking=True)
			self.y = y_t.to(self.device, non_blocking=True)

		# end

	def __len__(self):
		"""Ritorna la lunghezza del dataset.

		Returns:
			int: Numero di campioni nel dataset.
		"""

		return self.y.shape[0]
	
		# end

	def __getitem__(self, idx):
		"""Ritorna un elemento del dataset, spostato sul device se
        richiesto dal trasferimento lazy.

		Args:
			idx (int): Indice dell'elemento da recuperare.
		
		Returns:
			tuple: Coppia (X, y) del campione richiesto.
		"""
	
		x = self.X[idx]
		y = self.y[idx]

		if self.transform is not None:
			x = self.transform(x)

		if self.lazy_device_transfer:
			x = x.to(self.device, non_blocking=True)
			y = y.to(self.device, non_blocking=True)

		return x, y

		# end
	
	# end class

In [ ]:
#	FUNCTIONS
#   ####################################################################    #

def flow_debug(
		debug: bool = False,
		flows_to_inspect: list = None,
		metadata: pd.DataFrame = None,
		histograms: np.ndarray = None
	) -> None:

	"""Stampa informazioni di debug sui flussi specificati, se il flag di debug è attivo.
	Mostra le coordinate degli elementi non nulli, i valori e le statistiche di base
	come min, max, media, deviazione standard e somma.

	Args:
		debug (bool): Flag per abilitare la stampa delle informazioni di debug.
		flows_to_inspect (list): Lista degli ID dei flussi da ispezionare.
		metadata (pd.DataFrame): DataFrame contenente i metadati dei flussi.
		histograms (np.ndarray): Array NumPy contenente gli istogrammi 2D (FlowPic).
	"""

	if debug and flows_to_inspect is not None:
	
		for fid in flows_to_inspect:

			flow_metadata = metadata.loc[metadata['FlowID'] == fid]

			if flow_metadata.empty:
				print(f"[DEBUG] Flusso n.{fid} non trovato.")
				continue

			did = flow_metadata['DatasetID'].values[0]
			flow = histograms[did][0]

			#   ########################################################    #
			#	Stampa di shape e metadati del flusso

			print(f"[DEBUG] Elaborazione del flusso n.{fid}")
			print(f"Shape: {histograms[did].shape}")
			print(f"Metadata:\n{flow_metadata.to_string(index=False)}")

			#   ########################################################    #
			#	Stampa degli elementi non nulli
			#	Mostra le coordinate (riga, colonna) e il valore corrispondente.

			rows, cols = np.nonzero(flow)
			values = flow[rows, cols]

			print("Coordinate degli elementi non nulli:")
			for r, c, v in zip(rows, cols, values):
				print(f"[{r},{c}] = {v}", end=" | ")

			print()

			#   ########################################################    #
			#	Stampa delle statistiche di base

			print(f"Min: {np.min(histograms[did])}")
			print(f"Max: {np.max(histograms[did])}")
			print(f"Mean: {np.mean(histograms[did])}")
			print(f"Std: {np.std(histograms[did])}")
			print(f"Sum: {np.sum(histograms[did])}\n")

			# end for fid
		# end if

	# end

def get_flowpic_dir(data_path: str) -> str:
	""" Sostituisce il nome 'flowpic' al posto di 'mirage' nel percorso del dataset. """

	p = Path(data_path)
	parts = p.parts

	if "mirage" not in parts:
		raise ValueError(
			f"Il percorso '{data_path}' non contiene la cartella 'mirage'.\n"
			"Assicurati di fornire un percorso valido che contenga 'mirage'."
		)

	idx = parts.index("mirage")
	new_parts = list(parts)
	new_parts[idx] = "flowpic"

	return str(Path(*new_parts))

	# end

---

In [ ]:
importlib.reload(constants)
from constants import (
	USE_PRECOMPUTED_DATASET,
    DATA_PATH, DATASET_NAME,
    MIN_TPS, MIN_PACKETS, MIN_DIM
)

#   ####################################################################    #

# Si impostano i filtri per la selezione dei flussi da elaborare.
filters = {
	'min_tps': MIN_TPS,
	# 'min_dim': MIN_DIM,
	# 'min_packets': MIN_PACKETS,
}

# Si impostano le variabili di debug per il caricamento del dataset.
debug = False
debug_cycle = False
flows_to_inspect = None

print("Loading dataset...")

print(f"DEBUG impostato su {debug}.")
flowpics_name = get_dataset_name(DATA_PATH, DATASET_NAME, debug)
flowpics_dir = get_flowpic_dir(DATA_PATH)
os.makedirs(flowpics_dir, exist_ok=True)

npz_path = os.path.join(flowpics_dir, f"{flowpics_name}.npz")
meta_path = os.path.join(flowpics_dir, f"{flowpics_name}_metadata.csv")

print(f"USE_PRECOMPUTED_DATASET impostato su {USE_PRECOMPUTED_DATASET}.\n")
if not USE_PRECOMPUTED_DATASET:

	histograms, metadata = mirage_pickle_converter(
		f"{DATA_PATH}/{DATASET_NAME}",
		filters, debug, debug_cycle, flows_to_inspect
	)

	# Se il flag di debug è attivo,
	# si stampano informazioni dettagliate sui flussi specificati.
	flow_debug(debug, flows_to_inspect, metadata, histograms)

	# Salvataggio degli istogrammi 2D FlowPic in formato NumPy.
	np.savez_compressed(npz_path, data=histograms)
	print(f"Salvato dataset (shape={histograms.shape}) in: {npz_path}")

	# Salvataggio dei metadati in formato CSV.
	metadata.to_csv(meta_path, index=False)
	print(f"Salvati metadati (di {len(metadata)} righe) in: {meta_path} ")

elif USE_PRECOMPUTED_DATASET:
	if not os.path.exists(npz_path):
		raise FileNotFoundError(f"Array di istogrammi precomputati non trovato: {npz_path}")
	if not os.path.exists(meta_path):
		raise FileNotFoundError(f"DataFrame di metadati precomputati non trovato: {meta_path}")

	# Caricamento degli istogrammi 2D FlowPic precomputati.
	histograms = np.load(npz_path)['data']
	print(f"Caricato dataset (shape={histograms.shape}) da: {npz_path}")

	# Caricamento dei metadati in formato CSV.
	metadata = pd.read_csv(meta_path)
	print(f"Caricati metadati (di {len(metadata)} righe) da: {meta_path} ")

# Prima di procedere con split/training, si verifica che gli istogrammi e i metadati siano allineati.
assert histograms.shape[0] == len(metadata), "Disallineamento tra istogrammi e metadati!"
assert list(metadata["DatasetID"]) == list(range(len(metadata))), "DatasetID non contiguo!"

# Per coerenza con l'altro notebook, si rinominano le variabili per il dataset e le etichette.
X_raw = histograms
y_raw = metadata["Label"].values

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y_raw)

DATASET_CLASSES = le.classes_
""" List of unique classes in the dataset, determined by the unique labels in y_raw after encoding. """

NUM_CLASSES = len(le.classes_)
""" Number of unique classes in the dataset. """

print(
    f"Number of classes: {NUM_CLASSES} \n\n" +
	f"Classes: {DATASET_CLASSES}"
)

In [ ]:
importlib.reload(constants)
from constants import (
	TRAIN_SIZE, VAL_SIZE, TEST_SIZE,
    USE_NEW_SIZE, NEW_TRAIN_SIZE,
    RANDOM_SEED
)

#   ####################################################################    #

# Verifica che le proporzioni di Train, Val e Test sommino a "1".
assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == 1.0, "Train, Val, Test sizes must sum to 1."

# Il primo split divide il dataset in due parti: Train+Val e Test.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_encoded,
    test_size=TEST_SIZE,
    stratify=y_encoded,
    random_state=RANDOM_SEED
)

# Il fitting dello scaler viene fatto solo sul Train+Val, per evitare data leakage.
# ...

# Il secondo split divide il Train+Val rimanente in due insiemi: Train e Val.
tmp_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=tmp_size,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print(
	f"Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%}) \n" +
	f"Val shape:   {X_val.shape}   ({len(X_val)/len(X_raw):.1%}) \n" +
	f"Test shape:  {X_test.shape}  ({len(X_test)/len(X_raw):.1%}) \n"
)

# Riduzione della dimensione del training set, se necessario.
if USE_NEW_SIZE and len(X_train) > NEW_TRAIN_SIZE:
    
	# Calcola la frazione del training set da mantenere per ridurlo a NEW_TRAIN_SIZE
	percentage = NEW_TRAIN_SIZE / len(X_train)
    
	X_train, _, y_train, _ = train_test_split(
		X_train, y_train,
		train_size=percentage,
		stratify=y_train,
		random_state=RANDOM_SEED
	)
      
	print(f"Reduced Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%})")
      
else:
    
    print("Training set size is less than or equal to the new size. No reduction applied.")
    
	# end if

In [ ]:
importlib.reload(constants)
from constants import BATCH_SIZE

#   ####################################################################    #

# Si utilizza la classe dedicata MirageDataset per creare i dataset di PyTorch.
train_dataset = FlowPicDataset(X_train, y_train, device=DEVICE)
val_dataset = FlowPicDataset(X_val, y_val, device=DEVICE)
test_dataset = FlowPicDataset(X_test, y_test, device=DEVICE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False
)

In [ ]:
importlib.reload(constants)
from constants import (
	EXEC_MODE_TRAIN,
	OUTPUT_DIR, MODEL_TIMESTAMP_ID, MODEL_NAME,
    N_QUBITS, N_LAYERS,
    N_FEATURES, N_PACKETS
)

#   ####################################################################    #

SELECTED_MODEL = "FlowPicCNN"

HybridModel = get_model_class(SELECTED_MODEL)

model = HybridModel(
	n_qubits=N_QUBITS,
	n_layers=N_LAYERS,
	n_features=N_FEATURES,
	n_packets=N_PACKETS,
	num_classes=NUM_CLASSES
)

# Si sposta il modello sul dispositivo corretto (CPU, GPU o MPS)
# e si imposta il tipo di dato appropriato.
if DEVICE.type == 'mps':
	model = model.to(DEVICE).float()
else:
	model = model.to(DEVICE).double()

print(f"EXEC_MODE_TRAIN impostato su {EXEC_MODE_TRAIN}.\n")
if EXEC_MODE_TRAIN:
	print(f"Modello selezionato: {SELECTED_MODEL} -> {HybridModel.__name__}")
	print(model.get_model_name())
	print(model)

	total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
	print(f"Totale parametri addestrabili: {total_params:,}")

elif not EXEC_MODE_TRAIN:
	print(f"Modello selezionato: {MODEL_TIMESTAMP_ID} • {MODEL_NAME}")
	print("Il modello non verrà addestrato, ma solo valutato sul test set.")

	# Si caricano i pesi salvati del modello dalla directory di output specificata.
	model_dir = f"{OUTPUT_DIR}/{MODEL_TIMESTAMP_ID}/{MODEL_NAME}/model.pth"
	weights = torch.load(model_dir, map_location=DEVICE)

	# Verifica la compatibilità tra il modello corrente e i pesi salvati.
	check_model_compatibility(model, weights)

	# Carica i pesi nel modello corrente e stampa l'esito del caricamento.
	load_result = model.load_state_dict(weights)
	print(f"\nEsito caricamento: {load_result}")
	model = model.to(DEVICE)
	model.eval()

	print("Model loaded correctly.")

	# Si recupera lo storico delle metriche di addestramento e validazione dal file CSV salvato.
	df_history = pd.read_csv(f"{OUTPUT_DIR}/{MODEL_TIMESTAMP_ID}/{MODEL_NAME}/training_history.csv")
	history = df_history.to_dict(orient='list')

	# end if

In [ ]:
importlib.reload(constants)
from constants import (
	EXEC_MODE_TRAIN,
	LEARNING_RATE,
	LOSS_REGISTRY
)

#   ####################################################################    #

if not EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("La selezione della loss function e dell'optimizer è irrilevante "
	   "in questa modalità.")
	pass

elif EXEC_MODE_TRAIN:
	SELECTED_LOSS = "Focal"

	# Per completare la configurazione della fase di training,
	# si definisce l'optimizer e la funzione di loss.
	optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

	loss_dtype = torch.float if DEVICE.type == 'mps' else torch.double
	loss_params = LOSS_REGISTRY[SELECTED_LOSS]["params"]

	if SELECTED_LOSS == "CrossEntropy":
		criterion = nn.CrossEntropyLoss()

	elif SELECTED_LOSS == "WeightedCrossEntropy":
		class_weights = compute_class_weights(
			y_train,
			NUM_CLASSES
		).to(DEVICE).to(loss_dtype)

		criterion = nn.CrossEntropyLoss(weight=class_weights)

	elif SELECTED_LOSS == "Focal":
		alpha = loss_params.get("ALPHA", "class_weights")
		gamma = loss_params.get("GAMMA", 2.0)

		if alpha == "class_weights":
			alpha = compute_class_weights(y_train, NUM_CLASSES)
		elif alpha == "uniform":
			alpha = torch.ones(NUM_CLASSES)
		elif alpha == "custom":
			alpha = torch.tensor(alpha)
		else:
			raise ValueError(f"Invalid alpha_mode: {alpha}. Must be 'class_weights', 'uniform', or 'custom'.")

		alpha = alpha.to(DEVICE).to(loss_dtype)

		criterion = torch.hub.load(
			'adeelh/pytorch-multi-class-focal-loss',
			model='focal_loss',
			alpha=alpha,
			gamma=gamma,
			reduction='mean',
			device=DEVICE,
			dtype=loss_dtype,
			force_reload=False
		)

	else:
		raise ValueError(
			f"Tipo di loss non supportato: {SELECTED_LOSS}. "
			"Usare: 'CrossEntropy', 'WeightedCrossEntropy' oppure 'Focal'."
		)

	print(f"Loss selezionata: {SELECTED_LOSS}")

In [ ]:
importlib.reload(constants)
from constants import (
    EXEC_MODE_TRAIN,
	EPOCHS, PATIENCE, EARLY_STOPPING
)

#   ####################################################################    #

if not EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("Il modello non verrà addestrato.")
	pass

elif EXEC_MODE_TRAIN:
	# Questo dizionario terrà traccia delle metriche
	# di addestramento e validazione per ogni epoca.
	history = {
		'epoch': [],
		'time': [],
		'accuracy': [],
		'val_accuracy': [],
		'loss': [],
		'val_loss': [],
	}

	# Inizializza il timestamp per il salvataggio dei risultati e del modello.
	id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

	# Inizializza le variabili per il monitoraggio della miglior loss di validazione.
	best_val_loss = float('inf')
	best_model_wts = copy.deepcopy(model.state_dict())
	patience_counter = 0

	#   ####################################################################    #
	#	INIZIO TRAINING LOOP

	start_time = time.time()

	print(
		f"[INFO] Addestramento iniziato.\n" +
		f"[DATETIME] {id}\n" +
		f"[NAME] {model.get_model_name()}"
	)

	for epoch in range(EPOCHS):
		
		# Salva il timestamp di inizio epoca per calcolare la durata dell'epoca corrente.
		start_epoch_time = time.time()
		
		# Calcola la loss e l'accuratezza per il training set e il validation set.
		train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
		val_loss, val_acc = evaluate(model, val_loader, criterion)
		
		# Calcola la durata dell'epoca corrente.
		end_epoch_time = time.time()
		epoch_duration = end_epoch_time - start_epoch_time

		# Aggiorna lo storico delle metriche per l'epoca corrente.
		history['epoch'].append(epoch + 1)
		history['time'].append(epoch_duration)
		history['loss'].append(train_loss)
		history['accuracy'].append(train_acc)
		history['val_loss'].append(val_loss)
		history['val_accuracy'].append(val_acc)

		# Stampa in console le metriche dell'epoca corrente.
		print(f"Epoch {epoch+1}/{EPOCHS} | "
			f"Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | "
			f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}"
			f" | Time: {epoch_duration:.2f}s")

		# CHECK : se la loss di validazione migliora,
		# salva il modello e resetta il contatore per la patience.
		if val_loss < best_val_loss:
			best_val_loss = val_loss
			best_model_wts = copy.deepcopy(model.state_dict())
			patience_counter = 0
			print(f"  -> Validation loss improved. Model saved.")
		else:
			patience_counter += 1
			print(f"  -> No improvement. Patience: {patience_counter}", f"/ {PATIENCE}" if EARLY_STOPPING else "")

		# Se il contatore di patience raggiunge il limite
		# e l'early stopping è abilitato, si interrompe il training.
		if patience_counter >= PATIENCE and EARLY_STOPPING:
			print("Early stopping triggered.")
			break

		# end for epoch

	# Calcola e stampa il tempo totale di addestramento.
	total_time = time.time() - start_time
	print(f"\nTraining complete in {total_time/60:.2f} minutes.")

	# Salva i pesi del modello con la miglior loss di validazione.
	last_model = copy.deepcopy(model)
	model.load_state_dict(best_model_wts)

In [ ]:
importlib.reload(constants)
from constants import EXEC_MODE_TRAIN

#   ####################################################################    #

if not EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("Non è stato eseguito alcun addestramento. Il modello non verrà salvato.")
	pass

elif EXEC_MODE_TRAIN:
	model_name = f"{len(history['loss'])}E_{SELECTED_LOSS}_{SELECTED_MODEL}"
	output_dir = f"../results/{id}/"

	os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)

	# with open(f"{output_dir}/{model_name}/model_summary.txt", "w") as f:
	#     model.summary(print_fn=lambda x: f.write(x + "\n"))

	# Salva lo storico delle metriche di addestramento e validazione in un file CSV.
	df_history = pd.DataFrame(history)
	df_history.to_csv(f"{output_dir}/{model_name}/training_history.csv", index=False)

	# Salva i pesi del modello addestrato in un file .PTH
	torch.save(model.state_dict(), f"{output_dir}/{model_name}/model.pth")

In [ ]:
importlib.reload(constants)
from constants import EXEC_MODE_TRAIN

#   ####################################################################    #

if not EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("Non è stato eseguito alcun addestramento. Impossibile valutare il modello.")
	pass

elif EXEC_MODE_TRAIN:
	# Calcola la loss e l'accuratezza per il test set,
	# per valutare le prestazioni del modello su dati mai visti prima.
	test_loss, test_acc = evaluate(model, test_loader, criterion)
	print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
importlib.reload(constants)
from constants import SAVE_OUTPUT

#   ####################################################################    #

acc = history['accuracy']
val_acc = history['val_accuracy']
loss = history['loss']
val_loss = history['val_loss']
epochs_range = range(len(acc))

training_validation_plots = plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

if SAVE_OUTPUT:
	training_validation_plots.savefig(f"{output_dir}/{model_name}/train_val_plots.png")

In [ ]:
importlib.reload(constants)
from constants import SAVE_OUTPUT

#   ####################################################################    #

model.eval()
all_preds = []
all_labels = []

# Si utilizza "torch.no_grad()" per disabilitare
# il calcolo del gradiente durante la fase di valutazione,
# migliorando le prestazioni e riducendo l'uso della memoria.
with torch.no_grad():
    for inputs, labels in test_loader:
        
		# Sposta i dati sul DEVICE selezionato prima della predizione.
        inputs = inputs.to(DEVICE)
        
		# Calcola le predizioni del modello per il batch corrente.
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        
		# Infine, si utilizza "extend" per aggiungere le predizioni e le etichette
        # del batch corrente alle liste globali "all_preds" e "all_labels".
        # A differenza di "append", che aggiunge un singolo elemento,
		# "extend" aggiunge tutti gli elementi di un Iterable alla lista esistente.
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
		# ATTENZIONE!
		# Sostituire con "all_labels.extend(labels.numpy())" se si utilizza la CPU.
    
		# end for inputs, labels

cm = confusion_matrix(all_labels, all_preds, normalize='true')

confusion_matrix_plot = plt.figure(figsize=(10, 8))

ax = sns.heatmap(
	cm,
	annot=False,
	fmt='',
	cmap='plasma_r',   # Mappa colori plasma invertita
	linewidths=0.5,    # Spessore linee della griglia
	mask= cm == 0,     # Applica la maschera
	linecolor='black', # Colore linee della griglia
	square=True,       # Celle quadrate
	cbar_kws={ "ticks": [0.1, 1, 10, 100] },
	xticklabels=le.classes_,
	yticklabels=le.classes_
)

ax.set_facecolor('white')

empty_cols = np.where(cm.sum(axis=0) == 0)[0]

# Se ci sono colonne vuote, aggiunge un simbolo '•' al centro di ciascuna cella vuota.
for col in empty_cols:
    for row in range(cm.shape[0]):
        
        # Per centrare il testo nella cella, si aggiunge 0.5 a "row" e "col".
        ax.text(
            col + 0.5, row + 0.5, '•',
            ha='center', va='center',
            color='black', fontsize=15
		)
    
		# end for row
	# end for col

plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

if SAVE_OUTPUT:
	confusion_matrix_plot.savefig(f"{output_dir}/{model_name}/confusion_matrix.png")

In [ ]:
importlib.reload(constants)
from constants import SAVE_OUTPUT

#   ####################################################################    #

report_dict = classification_report(
    all_labels, all_preds,
    target_names=le.classes_, labels=np.arange(NUM_CLASSES),
    digits=4, zero_division=0
)

if SAVE_OUTPUT:
    with open(f"{output_dir}/{model_name}/classification_report.txt", "w") as f:
        f.write(report_dict)